# CAZ Interactive Demo

*Run Qwen2.5-7B on your own concept pairs and watch CAZ emerge*

This notebook loads **Qwen2.5-7B** (28 layers, 3584-dim hidden, GQA attention) and computes a full CAZ analysis on any concept you choose. Provide 10–20 contrastive sentence pairs and the notebook will show:
- Where in the model the concept crystallises (separation peak)
- Whether allocation is multimodal (two distinct assembly events)
- How your concept's peak depth compares to the 17 concepts from the paper

**Requirements:** GPU with ≥ 6 GB VRAM. Colab T4 (15 GB) works. Uses 4-bit quantization via `bitsandbytes`.

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026a)

In [ ]:
%pip install -q "rosetta_tools[extract,quantize]>=1.2.0"

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import hf_hub_download

from rosetta_tools.extraction import extract_contrastive_activations
from rosetta_tools.caz import compute_layer_metrics, find_caz_regions

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. Extraction will be slow on CPU — consider a smaller model.")

## 1. Load the model

We load **Qwen2.5-7B** (28 layers, 3584-dim hidden, GQA attention) in 4-bit. First download is ~4 GB; subsequent runs use the cached weights.

CAZ only needs `output_hidden_states=True`, which works identically on quantized models — quantization affects compute precision, not the geometric structure of the residual stream that we measure.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

n_layers = model.config.num_hidden_layers
print(f"Loaded: {MODEL_ID}")
print(f"  {n_layers} layers  |  {model.config.hidden_size}-dim hidden")

In [ ]:
# ── Alternative models — swap MODEL_ID above ─────────────────────────────────
# Pre-computed baseline results exist for all models marked (baseline ✓).
# Any HuggingFace base model works for extraction; only listed ones have the
# 17-concept baseline in the Rosetta Activations dataset for section 5.
#
# SMALL — fast on CPU or minimal GPU (baseline ✓)
#   MODEL_ID = "openai-community/gpt2"        # 12L,   768-dim,  ~500 MB
#   MODEL_ID = "openai-community/gpt2-large"  # 36L,  1280-dim,  ~1.5 GB
#   MODEL_ID = "openai-community/gpt2-xl"     # 48L,  1600-dim,  ~3 GB
#   MODEL_ID = "EleutherAI/pythia-70m"        # 6L,    512-dim,  ~150 MB
#   MODEL_ID = "EleutherAI/pythia-160m"       # 12L,   768-dim,  ~320 MB
#   MODEL_ID = "Qwen/Qwen2.5-0.5B"           # 24L,   896-dim,  ~1 GB
#   MODEL_ID = "Qwen/Qwen2.5-1.5B"           # 28L,  1536-dim,  ~3 GB
#
# MEDIUM — GPU recommended, 4-bit fits on Colab T4 (baseline ✓)
#   MODEL_ID = "Qwen/Qwen2.5-3B"             # 36L,  2048-dim,  ~6 GB (fp16)
#   MODEL_ID = "EleutherAI/pythia-2.8b"       # 32L,  2560-dim,  ~6 GB (fp16)
#   MODEL_ID = "meta-llama/Llama-3.2-3B"     # 28L,  3072-dim,  ~6 GB (fp16)
#   MODEL_ID = "microsoft/phi-2"              # 32L,  2560-dim,  ~5 GB (fp16)
#
# LARGE — 4-bit on Colab T4, default (baseline ✓)
#   MODEL_ID = "Qwen/Qwen2.5-7B"             # 28L,  3584-dim  ← DEFAULT
#   MODEL_ID = "google/gemma-2-9b"           # 42L,  3584-dim
#   MODEL_ID = "meta-llama/Llama-3.1-8B"     # 32L,  4096-dim
#   MODEL_ID = "mistralai/Mistral-7B-v0.3"   # 32L,  4096-dim
#   MODEL_ID = "EleutherAI/pythia-6.9b"      # 32L,  4096-dim
#
# VERY LARGE — needs A100/H100 or multi-GPU (baseline ✓)
#   MODEL_ID = "Qwen/Qwen2.5-14B"            # 48L,  5120-dim
#   MODEL_ID = "Qwen/Qwen2.5-32B"            # 64L,  5120-dim
#   MODEL_ID = "Qwen/Qwen2.5-72B"            # 80L,  8192-dim
#   MODEL_ID = "meta-llama/Llama-3.1-70B"    # 80L,  8192-dim
#   MODEL_ID = "tiiuae/falcon-40b"           # 60L,  8192-dim
#
# NOTE: section 5 baseline comparison uses Qwen2.5-7B results.
# For a different model, skip section 5 or load its baseline separately.

## 2. Define your concept

Edit the cell below:
- Set `CONCEPT_NAME` to a label for your concept
- Fill `pos_texts` with sentences that **clearly express** the concept
- Fill `neg_texts` with topically similar sentences that **lack** the concept

The contrast matters more than the size: 10–15 well-matched pairs work better than 30 loosely-matched ones. Keep `pos_texts` and `neg_texts` the same length.

The example below uses **causation** (from Paper 4). Swap it for anything you want to probe.

In [ ]:
# ── YOUR CONCEPT ─────────────────────────────────────────────────────────────
CONCEPT_NAME = "causation"

pos_texts = [
    "The bridge collapsed because the support beams had corroded.",
    "Scientists confirmed that the compound caused the cellular damage.",
    "Heavy rainfall led to widespread flooding across the valley.",
    "The fire spread rapidly due to the exceptionally dry conditions.",
    "Poor early nutrition resulted in slower cognitive development.",
    "The new austerity policy caused unemployment to rise sharply.",
    "Chronic stress triggered the patient's recurring migraines.",
    "The accident occurred as a direct result of driver fatigue.",
    "Her late arrival caused the entire meeting to be delayed.",
    "Prolonged exposure to the chemical produced severe respiratory burns.",
    "The burst pipe caused extensive water damage to the lower floors.",
    "Sleep deprivation led to serious errors in the team's judgement.",
]

neg_texts = [
    "The bridge collapsed, and the support beams were found to be corroded.",
    "Scientists observed the compound alongside the cellular damage.",
    "There was heavy rainfall, and the valley experienced widespread flooding.",
    "The conditions were exceptionally dry. The fire spread rapidly.",
    "The child had poor early nutrition and slower cognitive development.",
    "The new austerity policy was introduced. Unemployment rose sharply.",
    "The patient had chronic stress and recurring migraines.",
    "The driver was fatigued. The accident happened shortly after.",
    "She arrived late. The meeting was delayed.",
    "There was prolonged chemical exposure. Severe respiratory burns were present.",
    "The pipe burst. Water damage was found throughout the lower floors.",
    "The team members were sleep-deprived and made several serious errors.",
]
# ─────────────────────────────────────────────────────────────────────────────

assert len(pos_texts) == len(neg_texts), "pos and neg must be the same length"
print(f"Concept: {CONCEPT_NAME!r} — {len(pos_texts)} pairs")

## 3. Extract activations and compute CAZ

This runs all sentences through every layer of the model and computes $S(l)$, $C(l)$, $v(l)$ at each of the 28 layers. On a T4 GPU with 12 pairs, expect ~20–40 seconds.

In [ ]:
print("Extracting activations across all layers...")
layer_acts = extract_contrastive_activations(
    model, tokenizer, pos_texts, neg_texts,
    device=device, batch_size=4,
)

print("Computing CAZ metrics...")
metrics = compute_layer_metrics(layer_acts)
profile = find_caz_regions(metrics)
depth_pct = [m.layer / n_layers * 100 for m in metrics]

print(f"\nCAZ profile for '{CONCEPT_NAME}':")
print(f"  Allocation regions: {profile.n_regions}")
print(f"  Multimodal: {profile.is_multimodal}")
for i, region in enumerate(profile.regions):
    print(f"  Region {i+1}: peak layer {region.peak}  ({region.depth_pct:.1f}% depth)  "
          f"S={region.peak_separation:.3f}  C={region.peak_coherence:.3f}")

## 4. Visualise the allocation profile

The vertical dashed lines mark detected CAZ peaks. If there are two, the model is assembling the concept at two distinct depths — a shallow structural representation and a deeper semantic one.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
fig.suptitle(f"CAZ profile — '{CONCEPT_NAME}'  ({MODEL_ID.split('/')[-1]})",
             fontsize=13, fontweight="bold")

for ax, attr, ylabel, color in zip(
    axes,
    ["separation", "coherence", "velocity"],
    ["Separation  S(l)", "Coherence  C(l)", "Velocity  v(l)"],
    ["#1565C0", "#2E7D32", "#E65100"],
):
    vals = [getattr(m, attr) for m in metrics]
    ax.plot(depth_pct, vals, color=color, lw=2)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.axhline(0, color="#BDBDBD", lw=0.8, ls="--")
    ax.grid(alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for region in profile.regions:
        ax.axvline(region.depth_pct, color="#C62828", lw=1.5, ls=":", alpha=0.7)

axes[-1].set_xlabel("Depth (% of layers)", fontsize=11)
plt.tight_layout()
plt.show()

## 5. How does your concept compare?

The following cell downloads the pre-computed CAZ results for Qwen2.5-7B-Instruct across all 17 paper concepts and plots where your concept's peak lands on the shallow-to-deep ordering.

In [ ]:
HF_REPO = "james-ra-henry/Rosetta-Activations"
MODEL_KEY = "Qwen_Qwen2.5_7B"

PAPER_CONCEPTS = [
    "agency", "authorization", "causation", "certainty", "credibility",
    "deception", "exfiltration", "formality", "moral_valence", "negation",
    "plurality", "sarcasm", "sentiment", "specificity", "temporal_order",
    "threat_severity", "urgency",
]

baseline = {}
for c in PAPER_CONCEPTS:
    path = hf_hub_download(
        HF_REPO,
        filename=f"models/{MODEL_KEY}/caz_{c}.json",
        repo_type="dataset",
    )
    with open(path) as f:
        baseline[c] = json.load(f)["layer_data"]["peak_depth_pct"]

your_peak = profile.dominant.depth_pct

# Sort shallow → deep; highlight your concept if it's one of the 17
by_depth = sorted(baseline.items(), key=lambda x: x[1])
names = [c for c, _ in by_depth]
depths = [d for _, d in by_depth]

fig, ax = plt.subplots(figsize=(11, 4.5))
bar_colors = ["#EF5350" if c == CONCEPT_NAME else "#90CAF9" for c in names]
ax.barh(names, depths, color=bar_colors, height=0.65)

if CONCEPT_NAME not in baseline:
    ax.axvline(your_peak, color="#C62828", lw=2.5, ls="--",
               label=f"'{CONCEPT_NAME}' — your concept ({your_peak:.1f}%)")
    ax.legend(fontsize=10)

ax.set_xlabel("Peak depth (% of layers)", fontsize=11)
ax.set_title(f"Peak depth: your concept vs. 17 paper concepts  ({MODEL_ID.split('/')[-1]})",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

if CONCEPT_NAME in baseline:
    print(f"'{CONCEPT_NAME}' is a paper concept.  Paper value: {baseline[CONCEPT_NAME]:.1f}%   Your run: {your_peak:.1f}%")
else:
    rank = sum(1 for d in depths if d < your_peak) + 1
    print(f"'{CONCEPT_NAME}' peaks at {your_peak:.1f}%  —  rank {rank} of {len(depths)+1} (shallow → deep)")

## Next steps

- Try different concepts — does *uncertainty* behave differently from *certainty*? Does *irony* look like *sarcasm*?
- Swap `MODEL_ID` for any HuggingFace model: `extract_contrastive_activations` works with any model that exposes `output_hidden_states=True`
- **`03_caz_implementation_demo.ipynb`** — implement the metrics and Procrustes alignment from scratch, and reproduce the cross-architecture convergence result from Paper 4

| Paper | |
|-------|-|
| Paper 1 — CAZ Framework | [Henry 2026a](https://arxiv.org/abs/PLACEHOLDER) |
| Paper 4 — Cross-architecture PRH | [Henry 2026d](https://arxiv.org/abs/PLACEHOLDER) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |